# 12 · A durable Django consumer and safe replay

**Level:** advanced · **Time:** 50–70 minutes · **Needs:** running Kafka, notebook 11.

Build a materialized read model, simulate a crash after a database commit but before
an offset commit, and prove that replay changes the database only once. Then run the
same logic as a bounded Django management command and handle a poison event explicitly.

Restart the kernel, then Run All. A fresh SQLite database and topic isolate this run.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


### Jupyter and synchronous Django code

Jupyter kernels may keep an event loop running even with `%autoawait off`. The next cell
installs `%%django_sync`: it executes the visible Python body on one dedicated synchronous
thread and waits for completion. All Django code cells below use that thread, so ORM
connection ownership and transaction boundaries remain consistent. Variables stay in the
notebook namespace, exceptions remain visible, and Django's async safety stays enabled.
The magic adds no business logic. In a terminal, run the same Python bodies normally.


In [ ]:
%autoawait off
sys.path.insert(0, str(ROOT / "django_demo"))
from notebook_support import install_sync_magic
DJANGO_EXECUTOR = install_sync_magic(get_ipython())
print("Synchronous Django cell executor installed")


In [ ]:
%%django_sync
LESSON = "12"

import os
import django
from django.conf import settings

if settings.configured:
    raise RuntimeError("Restart the kernel before running this notebook again; Django settings are process-wide.")
DATA_DIR = ROOT / ".lab-data"
DATA_DIR.mkdir(exist_ok=True)
DB_PATH = DATA_DIR / f"django-{LESSON}-{RUN}.sqlite3"
os.environ["DJANGO_DB_PATH"] = str(DB_PATH)
os.environ["DJANGO_SETTINGS_MODULE"] = "config.settings"
sys.path.insert(0, str(ROOT / "django_demo"))
django.setup()
from django.core.management import call_command
from django.db import transaction
from django.test import Client, override_settings
from django.utils import timezone
from decimal import Decimal
from unittest.mock import patch
from orders.models import Order, OutboxEvent, InboxEvent, OrderProjection
from orders.services import (
    order_event, validate_event, publish_event, apply_event,
    create_order_after_commit, create_order_with_outbox, publish_pending,
)
call_command("migrate", verbosity=0)
http = Client()
print("Isolated database:", DB_PATH.name)


## Step 1 — Create one event

The source order and the projection represent different responsibilities even though
this compact workshop keeps their tables in one SQLite database. In a deployed service
they may live in separate databases. The consumer needs only the event's contents.


In [ ]:
%%django_sync
require_broker()
topic = new_topic("django-consumer", partitions=1)
group = f"lab.django-consumer.{RUN}"
order = Order.objects.create(customer="Grace", total=Decimal("12.50"))
event = order_event(order)
publish_event(event, topic)
print("Event:", event["event_id"], "group:", group)


## Step 2 — Inspect atomic deduplication

`InboxEvent.event_id` is a primary key. The receipt and the projection are committed in
one database transaction. If the projection write fails, the receipt rolls back too.
An existing receipt means the event was already applied. Reusing an ID with different
content is rejected rather than silently accepted.


In [ ]:
%%django_sync
import inspect
print(inspect.getsource(apply_event))


## Step 3 — Process a record, then omit the offset commit

Closing this consumer without committing simulates the Kafka-visible result of a
process crash after the database commit. Automatic offset commits are disabled.
No background thread can acknowledge this record on our behalf.


In [ ]:
%%django_sync
first = consumer(group)
try:
    first.subscribe([topic])
    original_message = poll_records(first, count=1)[0]
    assert apply_event(decode(original_message)) is True
    original_offset = original_message.offset()
    # Deliberately no first.commit(...).
finally:
    first.close()
assert InboxEvent.objects.count() == 1
assert OrderProjection.objects.count() == 1
print("Database committed; Kafka offset was not committed")


## Step 4 — Restart with the SAME group

The group has no committed offset, so `earliest` selects the record again. The second
application sees the durable receipt and skips the side effect. Only then does it
commit the next offset. Kafka offsets identify positions, not business event IDs.


In [ ]:
%%django_sync
restarted = consumer(group)
try:
    restarted.subscribe([topic])
    replayed = poll_records(restarted, count=1)[0]
    assert replayed.offset() == original_offset
    assert apply_event(decode(replayed)) is False
    restarted.commit(message=replayed, asynchronous=False)
    from confluent_kafka import TopicPartition
    committed = restarted.committed([TopicPartition(topic, 0)], timeout=10)[0].offset
    assert committed == replayed.offset() + 1
finally:
    restarted.close()
assert InboxEvent.objects.count() == OrderProjection.objects.count() == 1
print("Replay was harmless; committed next offset:", committed)


## Step 5 — Query the read model through HTTP

Reading the projection does not call Kafka. This is eventual consistency: the write
request may finish before its read model appears. A 404 can mean 'not projected yet'.


In [ ]:
%%django_sync
projection = http.get(f"/projections/{order.id}/")
assert projection.status_code == 200
assert projection.json()["total"] == "12.50"
print(projection.json())


## Step 6 — Run the same consumer as a separate command

Create two more events, then launch a real child process with the same database path.
The command exits after two messages or 20 seconds. There are no worker threads inside
Django's web server or `AppConfig.ready()`.


In [ ]:
%%django_sync
more_events = []
for customer, total in [("Linus", "8.00"), ("Margaret", "13.00")]:
    source = Order.objects.create(customer=customer, total=Decimal(total))
    more_events.append(order_event(source))
    publish_event(more_events[-1], topic)
import subprocess
completed = subprocess.run(
    [sys.executable, str(ROOT / "django_demo" / "manage.py"), "consume_orders",
     "--topic", topic, "--group", group, "--max-messages", "2", "--timeout", "20"],
    cwd=ROOT, env=os.environ.copy(), text=True, capture_output=True, timeout=45, check=True,
)
print(completed.stdout)
assert OrderProjection.objects.count() == 3


## Step 7 — Observe a duplicate arriving at a NEW Kafka offset

A duplicate business event can be a completely new Kafka record. This is common when
an outbox publisher crashes after delivery but before marking a row sent. Producer
idempotence does not identify a second application-level publish as the same event.


In [ ]:
%%django_sync
publish_event(event, topic)
call_command("consume_orders", topic=topic, group=group, max_messages=1, timeout=20)
assert OrderProjection.objects.count() == 3
assert InboxEvent.objects.count() == 3
print("Duplicate event at another offset left the projection unchanged")


## Step 8 — Make poison handling explicit

Send an unsupported schema version. This consumer uses a **fail-stop policy**: it
reports topic/partition/offset, closes, and does not commit the bad record. That preserves
evidence and blocks subsequent progress on this partition until an operator repairs,
quarantines, or deliberately skips it. Notebook 08 demonstrates a dead-letter alternative.


In [ ]:
%%django_sync
import copy
poison = copy.deepcopy(event)
poison["event_id"] = str(uuid4())
poison["schema_version"] = 999
publish(topic, [poison])  # Generic publisher deliberately bypasses the domain validator.
from django.core.management.base import CommandError
try:
    call_command("consume_orders", topic=topic, group=group, max_messages=1, timeout=20)
except CommandError as exc:
    print("Expected stop:", exc)
else:
    raise AssertionError("The poison record should stop this consumer")
assert InboxEvent.objects.count() == 3


## Step 9 — Verify the poison record remains unacknowledged

A separate consumer using the same group receives the poison record again. Inspecting
it does not advance the committed offset. In a real incident, avoid repeated retries
without a plan: decide who owns schema repair and where rejected events are retained.


In [ ]:
%%django_sync
diagnostic = consumer(group)
try:
    diagnostic.subscribe([topic])
    stuck = poll_records(diagnostic, count=1)[0]
    assert decode(stuck)["schema_version"] == 999
    previous_commit = diagnostic.committed([TopicPartition(topic, 0)], timeout=10)[0].offset
    assert previous_commit == stuck.offset()
    print("Committed next offset:", previous_commit, "poison offset:", stuck.offset())
finally:
    diagnostic.close()


## Step 10 — Prove receipts cannot outlive failed side effects

Inject a database-side error after receipt creation. The surrounding `atomic()` block
rolls back both operations, so a later retry is still allowed to apply the event.


In [ ]:
%%django_sync
source = Order.objects.create(customer="Retry safely", total=Decimal("4.00"))
retryable = order_event(source)
with patch("orders.services.OrderProjection.objects.create", side_effect=RuntimeError("simulated database failure")):
    try:
        apply_event(retryable)
    except RuntimeError:
        pass
assert not InboxEvent.objects.filter(event_id=retryable["event_id"]).exists()
assert apply_event(retryable) is True
print("Receipt rolled back; retry applied successfully")


## Practice and solution hints

1. Explain why committing Kafka before updating the database can lose a side effect.
2. Replay the topic into a new group. Why do existing receipt rows still suppress duplicates?
3. Design receipt retention. How long might historical events be replayed?
4. Extend the consumer with a DLQ. Which acknowledgement must succeed before source commit?

**Hints:** database first + Kafka commit second creates a replay window, handled by durable
deduplication. A new group changes Kafka offsets, not database history. Receipt retention
must cover replay requirements. A DLQ publish must be acknowledged before committing the
source, and DLQ consumers must tolerate duplicates too. This is at-least-once delivery with
idempotent database effects, not a distributed exactly-once transaction.
